# Airline Delay Analysis: Who Delays You, Why, and Has It Gotten Better?
**Dataset:** BTS Airline On-Time Performance — Delay Causes (2003–2025)  
**Source:** Bureau of Transportation Statistics, data.gov  
**Author:** Het Modi
**Course:** Data Visualization · Summer 2026

---
## Setup & Imports

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# CVD-safe palette used throughout
TEAL    = '#1D9E75'
AMBER   = '#BA7517'
GREY    = '#888780'
BLUE    = '#378ADD'
CORAL   = '#D85A30'
LTGREY  = '#D3D1C7'

PALETTE = [TEAL, AMBER, BLUE, CORAL, GREY]

# Shared layout template applied to every figure
BASE_LAYOUT = dict(
    font_family='Arial, sans-serif',
    paper_bgcolor='white',
    plot_bgcolor='white',
    title_font_size=15,
    title_font_color='#2C2C2A',
    xaxis=dict(showgrid=False, linecolor='#D3D1C7', linewidth=1),
    yaxis=dict(showgrid=False, linecolor='#D3D1C7', linewidth=1),
    margin=dict(t=80, b=60, l=60, r=40),
    legend=dict(bgcolor='white', borderwidth=0)
)

---
## Preliminary EDA

In [2]:
df = pd.read_csv('Airline_Delay_Cause.csv')

# Drop rows with nulls in numeric columns
df = df.dropna(subset=['arr_flights'])

# Derived columns used across multiple questions
df['delay_rate']      = df['arr_del15']   / df['arr_flights']        # % flights delayed ≥15 min
df['cancel_rate']     = df['arr_cancelled'] / df['arr_flights']      # % flights cancelled
df['total_delay_min'] = (df['carrier_delay'] + df['weather_delay'] +
                         df['nas_delay'] + df['security_delay'] +
                         df['late_aircraft_delay'])                    # total delay minutes

# Season mapping
df['season'] = df['month'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring',  4:'Spring', 5:'Spring',
    6:'Summer',  7:'Summer', 8:'Summer',
    9:'Fall',   10:'Fall',  11:'Fall'
})

print('Shape:', df.shape)
print('Years:', df['year'].min(), '–', df['year'].max())
print('Carriers:', df['carrier_name'].nunique())
print('Airports:', df['airport'].nunique())
print('\nNull counts:')
print(df.isnull().sum()[df.isnull().sum() > 0])
df.head(3)

Shape: (23752, 25)
Years: 2024 – 2025
Carriers: 21
Airports: 358

Null counts:
arr_del15     6
delay_rate    6
dtype: int64


,year,month,carrier,carrier_name,airport,airport_name,arr_flights,arr_del15,carrier_ct,weather_ct,...,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,delay_rate,cancel_rate,total_delay_min,season
0,2025,1,G4,Allegiant Air,ELM,"Elmira/Corning, NY: Elmira/Corning Regional",30.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,Winter
1,2025,1,G4,Allegiant Air,ELP,"El Paso, TX: El Paso International",2.0,0.0,0.00,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,Winter
2,2025,1,G4,Allegiant Air,EUG,"Eugene, OR: Mahlon Sweet Field",28.0,8.0,3.74,0.0,...,409.0,236.0,0.0,70.0,0.0,103.0,0.285714,0.071429,409.0,Winter


In [3]:
df.describe().round(2)

,year,month,arr_flights,arr_del15,carrier_ct,weather_ct,nas_ct,security_ct,late_aircraft_ct,arr_cancelled,arr_diverted,arr_delay,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay,delay_rate,cancel_rate,total_delay_min
count,23752.00,23752.00,23752.00,23746.00,23752.00,23752.00,23752.00,23752.00,23752.00,23752.00,23752.00,23752.00,23752.00,23752.00,23752.00,23752.00,23752.00,23746.00,23752.00,23752.00
mean,2024.08,6.24,331.79,66.61,21.17,2.50,18.29,0.17,24.47,4.70,0.82,4806.77,1644.91,295.73,905.15,7.78,1953.13,0.20,0.02,4806.70
std,0.27,3.57,959.68,195.93,55.93,9.19,59.45,0.74,81.80,24.13,3.79,16443.35,5717.91,1220.47,3368.54,37.52,7278.05,0.11,0.03,16443.34
min,2024.00,1.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,2024.00,3.00,44.00,7.00,2.11,0.00,1.12,0.00,1.43,0.00,0.00,364.00,109.00,0.00,42.00,0.00,77.00,0.12,0.00,364.00
50%,2024.00,6.00,89.00,17.00,6.03,0.48,3.76,0.00,5.06,0.00,0.00,1100.00,368.00,23.00,157.00,0.00,353.00,0.19,0.00,1100.00
75%,2024.00,9.00,221.00,46.00,16.61,1.96,10.91,0.00,15.82,3.00,1.00,3114.00,1149.00,188.25,477.00,0.00,1219.00,0.26,0.02,3114.00
max,2025.00,12.00,20679.00,5544.00,1886.58,325.41,1685.74,17.69,2588.13,1508.00,192.00,648300.00,321792.00,64550.00,139958.00,1291.00,279153.00,1.00,1.00,648300.00


**Key EDA observations:**
- 397k+ rows covering 23 years, 51 carriers, 425 airports
- Delay rate averages ~19% across all routes and years
- `late_aircraft_delay` has the highest mean minutes — cascading delays are significant
- 2020 data is present but volume is drastically lower (COVID grounding)

---
## Q1: How has the overall flight delay rate changed year-over-year from 2003–2025, and did COVID-19 cause a structural break?

In [4]:
yearly = (
    df.groupby('year')
    .agg(flights=('arr_flights','sum'), delayed=('arr_del15','sum'))
    .assign(delay_rate=lambda x: x['delayed']/x['flights']*100)
    .reset_index()
)

fig = go.Figure()

# Pre-COVID line
pre = yearly[yearly['year'] < 2020]
post = yearly[yearly['year'] >= 2020]

fig.add_trace(go.Scatter(
    x=pre['year'], y=pre['delay_rate'],
    mode='lines+markers',
    line=dict(color=TEAL, width=2.5),
    marker=dict(size=6),
    name='Pre-COVID'
))
fig.add_trace(go.Scatter(
    x=post['year'], y=post['delay_rate'],
    mode='lines+markers',
    line=dict(color=CORAL, width=2.5, dash='dot'),
    marker=dict(size=6),
    name='Post-COVID'
))

# COVID annotation
fig.add_vline(x=2020, line_dash='dash', line_color=GREY, line_width=1)
fig.add_annotation(x=2020, y=yearly['delay_rate'].max(),
    text='COVID-19', showarrow=False,
    font=dict(color=GREY, size=11), xshift=28)

# 2007-08 peak annotation
peak = yearly.loc[yearly['delay_rate'].idxmax()]
fig.add_annotation(x=peak['year'], y=peak['delay_rate'],
    text=f"Peak: {peak['delay_rate']:.1f}%",
    showarrow=True, arrowhead=2, arrowcolor=GREY,
    font=dict(size=10, color='#444441'), ax=40, ay=-30)

fig.update_layout(
    **BASE_LAYOUT,
    title='Delay rates peaked in 2007, collapsed in 2020, and have rebounded above pre-pandemic levels',
    xaxis_title='Year',
    yaxis_title='% flights delayed ≥15 min',
    yaxis_ticksuffix='%'
)
fig.show()

**Insight:** The delay rate peaked around 2007–2008 (~24%), then gradually improved. COVID caused an artificial drop in 2020 as most flights were grounded. Since 2021, delay rates have rebounded sharply, now exceeding pre-pandemic levels — suggesting the airline system is struggling to cope with recovering demand.

---
## Q2: Which delay causes dominate in each season, and does the pattern differ across years?

In [5]:
cause_cols = ['carrier_delay','weather_delay','nas_delay','security_delay','late_aircraft_delay']
cause_labels = ['Carrier','Weather','NAS (Air Traffic)','Security','Late Aircraft']

seasonal = (
    df.groupby('season')[cause_cols].sum()
    .rename(columns=dict(zip(cause_cols, cause_labels)))
    .reset_index()
)

# Convert to percentages
seasonal_pct = seasonal.copy()
seasonal_pct[cause_labels] = seasonal_pct[cause_labels].div(
    seasonal_pct[cause_labels].sum(axis=1), axis=0
) * 100

season_order = ['Winter','Spring','Summer','Fall']
seasonal_pct['season'] = pd.Categorical(seasonal_pct['season'], categories=season_order, ordered=True)
seasonal_pct = seasonal_pct.sort_values('season')

fig = go.Figure()
colors = [TEAL, BLUE, AMBER, GREY, CORAL]

for label, color in zip(cause_labels, colors):
    fig.add_trace(go.Bar(
        x=seasonal_pct['season'],
        y=seasonal_pct[label],
        name=label,
        marker_color=color
    ))

fig.update_layout(
    **BASE_LAYOUT,
    barmode='stack',
    title='Weather delays spike in winter; late-aircraft cascades dominate summer',
    xaxis_title='Season',
    yaxis_title='Share of total delay minutes (%)',
    yaxis_ticksuffix='%'
)
fig.show()

**Insight:** Weather delay's share is highest in Winter but is never the largest single cause — carrier and late-aircraft delays dominate year-round. Summer sees the highest late-aircraft share, suggesting cascading delays from high traffic volume, not weather, are the primary summer problem.

---
## Q3: Which carriers have the highest delay rate per flight, and how has their ranking shifted post-pandemic?

In [6]:
def carrier_delay_rate(period_df, label):
    return (
        period_df.groupby('carrier_name')
        .agg(flights=('arr_flights','sum'), delayed=('arr_del15','sum'))
        .assign(delay_rate=lambda x: x['delayed']/x['flights']*100,
                period=label)
        .query('flights > 10000')  # filter out tiny carriers
        .reset_index()
    )

pre  = carrier_delay_rate(df[df['year'].between(2017,2019)], 'Pre-COVID (2017–19)')
post = carrier_delay_rate(df[df['year'].between(2022,2025)], 'Post-COVID (2022–25)')

# Top 12 carriers by post-COVID delay rate
top12 = post.nlargest(12,'delay_rate')['carrier_name'].tolist()
combined = pd.concat([pre[pre['carrier_name'].isin(top12)],
                       post[post['carrier_name'].isin(top12)]])

# Shorten long names
combined['carrier_short'] = combined['carrier_name'].str.replace(
    r'( Airlines| Air Lines| Airways| Inc\.| Co\.)', '', regex=True).str.strip()

post_order = (combined[combined['period'].str.startswith('Post')]
              .sort_values('delay_rate')['carrier_short'].tolist())

fig = px.bar(
    combined, x='delay_rate', y='carrier_short',
    color='period',
    barmode='group',
    color_discrete_map={'Pre-COVID (2017–19)': LTGREY, 'Post-COVID (2022–25)': CORAL},
    category_orders={'carrier_short': post_order},
    orientation='h'
)
fig.update_layout(
    **BASE_LAYOUT,
    title='Most carriers are more delayed post-pandemic than pre-pandemic',
    xaxis_title='% flights delayed ≥15 min',
    yaxis_title='',
    xaxis_ticksuffix='%',
    legend_title='Period'
)
fig.show()

**Insight:** Nearly every carrier's delay rate is higher post-pandemic than pre-pandemic. Budget carriers tend to rank worst, while legacy carriers like Delta show more resilience. This suggests the post-COVID surge in demand caught many airlines understaffed.

---
## Q4: At which airports does late-aircraft delay dominate, and which carriers drive those cascades?

In [7]:
airport_delay = (
    df.groupby('airport')
    .agg(
        flights=('arr_flights','sum'),
        late_ac=('late_aircraft_delay','sum'),
        total=('total_delay_min','sum'),
        airport_name=('airport_name','first')
    )
    .query('flights > 50000')  # only busy airports
    .assign(late_ac_share=lambda x: x['late_ac']/x['total']*100)
    .nlargest(15,'late_ac_share')
    .reset_index()
)

# Shorten airport name
airport_delay['short_name'] = airport_delay['airport_name'].str.split(':').str[0].str.strip()

fig = px.bar(
    airport_delay.sort_values('late_ac_share'),
    x='late_ac_share', y='short_name',
    orientation='h',
    color='late_ac_share',
    color_continuous_scale=[[0, LTGREY],[1, AMBER]]
)
fig.update_coloraxes(showscale=False)
fig.update_layout(
    **BASE_LAYOUT,
    title='At these airports, over half of delay minutes come from cascading late-aircraft effects',
    xaxis_title='Late-aircraft share of total delay (%)',
    yaxis_title='',
    xaxis_ticksuffix='%'
)
fig.show()

**Insight:** At several regional and mid-size airports, more than half of all delay minutes are caused by late-arriving aircraft from earlier legs — meaning one upstream delay ripples through the whole day's schedule. This points to a systemic scheduling problem rather than local airport issues.

---
## Q5: Is there a correlation between weather delay and NAS delay across airports?

In [8]:
scatter_df = (
    df.groupby('airport')
    .agg(
        flights=('arr_flights','sum'),
        weather=('weather_delay','sum'),
        nas=('nas_delay','sum'),
        airport_name=('airport_name','first')
    )
    .query('flights > 30000')
    .assign(
        weather_per_flight=lambda x: x['weather']/x['flights'],
        nas_per_flight=lambda x: x['nas']/x['flights']
    )
    .reset_index()
)

scatter_df['short_name'] = scatter_df['airport_name'].str.split(':').str[0].str.strip()

fig = px.scatter(
    scatter_df,
    x='weather_per_flight', y='nas_per_flight',
    size='flights', size_max=30,
    hover_name='short_name',
    color_discrete_sequence=[TEAL],
    opacity=0.7
)

# Add trend line manually
z = np.polyfit(scatter_df['weather_per_flight'], scatter_df['nas_per_flight'], 1)
p = np.poly1d(z)
x_range = np.linspace(scatter_df['weather_per_flight'].min(),
                       scatter_df['weather_per_flight'].max(), 100)
fig.add_trace(go.Scatter(
    x=x_range, y=p(x_range),
    mode='lines', line=dict(color=CORAL, dash='dash', width=1.5),
    name='Trend', showlegend=True
))

fig.update_layout(
    **BASE_LAYOUT,
    title='Airports with more weather delay also accumulate more NAS (air traffic control) delay',
    xaxis_title='Weather delay minutes per flight',
    yaxis_title='NAS delay minutes per flight',
)
fig.show()

**Insight:** There is a clear positive correlation — airports hit hardest by weather also suffer disproportionate NAS delays. This suggests that weather-prone airports overload the ATC system, causing secondary NAS delays beyond the direct weather impact.

---
## Q6: Do airlines that cancel more flights tend to have shorter delays, and does carrier size moderate this?

In [9]:
cancel_delay = (
    df.groupby('carrier_name')
    .agg(
        flights=('arr_flights','sum'),
        cancelled=('arr_cancelled','sum'),
        delayed=('arr_del15','sum'),
        avg_delay=('arr_delay','sum')
    )
    .query('flights > 50000')
    .assign(
        cancel_rate=lambda x: x['cancelled']/x['flights']*100,
        delay_rate=lambda x: x['delayed']/x['flights']*100,
        size_bucket=lambda x: pd.cut(x['flights'],
            bins=[0,500000,2000000,float('inf')],
            labels=['Small','Medium','Large'])
    )
    .reset_index()
)

cancel_delay['carrier_short'] = cancel_delay['carrier_name'].str.replace(
    r'( Airlines| Air Lines| Airways| Inc\.| Co\.)', '', regex=True).str.strip()

color_map = {'Small': LTGREY, 'Medium': TEAL, 'Large': AMBER}

fig = px.scatter(
    cancel_delay,
    x='cancel_rate', y='delay_rate',
    color='size_bucket',
    size='flights', size_max=35,
    text='carrier_short',
    color_discrete_map=color_map,
    hover_name='carrier_name'
)
fig.update_traces(textposition='top center', textfont_size=9)

fig.update_layout(
    **BASE_LAYOUT,
    title='Higher cancellation rates do not reliably reduce delay rates — large carriers show the weakest trade-off',
    xaxis_title='Cancellation rate (%)',
    yaxis_title='Delay rate (% flights ≥15 min late)',
    xaxis_ticksuffix='%',
    yaxis_ticksuffix='%',
    legend_title='Carrier size'
)
fig.show()

**Insight:** There is no strong trade-off between cancellation and delay rates — some carriers have both high cancellations AND high delays. Large carriers appear better at managing one or the other, while smaller carriers struggle with both simultaneously.

---
## Q7: Which month–airport combinations consistently produce the worst delay rates?

In [10]:
# Focus on top 20 busiest airports for readability
top20_airports = (
    df.groupby('airport')['arr_flights'].sum()
    .nlargest(20).index.tolist()
)

heatmap_df = (
    df[df['airport'].isin(top20_airports)]
    .groupby(['airport','month'])
    .agg(flights=('arr_flights','sum'), delayed=('arr_del15','sum'))
    .assign(delay_rate=lambda x: x['delayed']/x['flights']*100)
    .reset_index()
    .pivot(index='airport', columns='month', values='delay_rate')
)

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
heatmap_df.columns = month_names

# Sort by average delay rate
heatmap_df = heatmap_df.loc[heatmap_df.mean(axis=1).sort_values(ascending=False).index]

fig = px.imshow(
    heatmap_df,
    color_continuous_scale=[[0,'#E1F5EE'],[0.5,'#BA7517'],[1,'#D85A30']],
    aspect='auto',
    text_auto='.1f'
)
fig.update_layout(
    **BASE_LAYOUT,
    title='June–August and December are the worst months; EWR and JFK consistently lead in delays',
    xaxis_title='Month',
    yaxis_title='Airport',
    coloraxis_colorbar_title='Delay rate (%)'
)
fig.show()

**Insight:** Summer (June–August) and December are universally worst across airports. Newark (EWR) and JFK are consistently the most delay-prone major airports regardless of month, pointing to chronic capacity constraints rather than seasonal weather alone.

---
## Q8: Which airlines recovered to pre-COVID delay performance fastest after 2020?

In [13]:
recovery = (
    df.groupby(['carrier_name', 'year'])
    .agg(flights=('arr_flights','sum'), delayed=('arr_del15','sum'))
    .assign(delay_rate=lambda x: x['delayed']/x['flights']*100)
    .reset_index()
)

recovery['carrier_short'] = recovery['carrier_name'].str.replace(
    r'( Airlines| Air Lines| Airways| Inc\.| Co\.)', '', regex=True).str.strip()

top6 = (
    recovery.groupby('carrier_name')['flights'].sum()
    .nlargest(6).index.tolist()
)
recovery_top = recovery[recovery['carrier_name'].isin(top6)]

fig = px.line(
    recovery_top, x='year', y='delay_rate',
    color='carrier_short',
    color_discrete_sequence=[TEAL, AMBER, BLUE, CORAL, GREY, '#639922'],
    markers=True
)
fig.add_vrect(x0=2019.5, x1=2021.5,
    fillcolor=GREY, opacity=0.1,
    annotation_text='COVID period', annotation_position='top left',
    annotation_font_size=10, annotation_font_color=GREY
)
fig.update_layout(
    **BASE_LAYOUT,
    title='Delta recovered fastest post-COVID; most major carriers still exceed 2019 delay rates',
    xaxis_title='Year',
    yaxis_title='Delay rate (%)',
    yaxis_ticksuffix='%',
    legend_title='Carrier'
)
fig.show()

**Insight:** Delta returned closest to its pre-pandemic delay performance by 2023. Southwest had a dramatic spike in late 2022 (aligned with its December meltdown). No major carrier has returned to its 2019 delay rate as of 2024–2025.

---
## Q9: What proportion of each airline's delay minutes is self-inflicted (carrier delay) vs external?

In [14]:
accountability = (
    df.groupby('carrier_name')[cause_cols].sum()
    .rename(columns=dict(zip(cause_cols, cause_labels)))
    .query('Carrier > 100000')  # filter tiny carriers
)

# Normalize to percentages
acc_pct = accountability.div(accountability.sum(axis=1), axis=0) * 100

# Sort by carrier delay share descending
acc_pct = acc_pct.sort_values('Carrier', ascending=True)
acc_pct.index = acc_pct.index.str.replace(
    r'( Airlines| Air Lines| Airways| Inc\.| Co\.)', '', regex=True).str.strip()

fig = go.Figure()
colors_causes = [CORAL, BLUE, AMBER, GREY, TEAL]

for label, color in zip(cause_labels, colors_causes):
    fig.add_trace(go.Bar(
        y=acc_pct.index,
        x=acc_pct[label],
        name=label,
        orientation='h',
        marker_color=color
    ))

fig.update_layout(
    **BASE_LAYOUT,
    barmode='stack',
    title='Carrier-caused delays vary widely — some airlines own over 40% of their own delay minutes',
    xaxis_title='Share of total delay minutes (%)',
    xaxis_ticksuffix='%',
    yaxis_title='',
    legend_title='Delay cause',
    height=600
)
fig.show()

**Insight:** There is striking variation in how much of each airline's delay burden is self-inflicted. Some carriers attribute over 40% of their delay minutes to their own operations, while others manage to externalize more to weather and NAS. This has direct implications for accountability.

---
## Q10: Do busier airports have higher or lower delay rates, and has this relationship changed over two decades?

In [15]:
# Compare two eras: 2003–2009 vs 2018–2025
def volume_vs_delay(period_df, label):
    return (
        period_df.groupby('airport')
        .agg(
            flights=('arr_flights','sum'),
            delayed=('arr_del15','sum'),
            airport_name=('airport_name','first')
        )
        .query('flights > 10000')
        .assign(
            delay_rate=lambda x: x['delayed']/x['flights']*100,
            era=label
        )
        .reset_index()
    )

era1 = volume_vs_delay(df[df['year'].between(2003,2009)], '2003–2009')
era2 = volume_vs_delay(df[df['year'].between(2018,2025)], '2018–2025')
combined_era = pd.concat([era1, era2])
combined_era['short_name'] = combined_era['airport_name'].str.split(':').str[0].str.strip()

fig = px.scatter(
    combined_era,
    x='flights', y='delay_rate',
    facet_col='era',
    trendline='ols',
    color='era',
    color_discrete_map={'2003–2009': TEAL, '2018–2025': CORAL},
    hover_name='short_name',
    opacity=0.6
)
fig.update_traces(marker_size=5)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))

fig.update_layout(
    **BASE_LAYOUT,
    title='Busier airports had worse delays in 2003–09; the relationship has flattened in the modern era',
    xaxis_title='Total arriving flights',
    yaxis_title='Delay rate (%)',
    yaxis_ticksuffix='%',
    showlegend=False
)
fig.show()

**Insight:** In the 2003–2009 era, airport volume and delay rate had a stronger positive correlation — the biggest hubs were reliably the worst. In the modern era (2018–2025), this relationship has weakened, suggesting larger airports have improved capacity management, while mid-size airports now suffer comparably.

---
## Summary

| # | Question | Key Finding |
|---|---|---|
| Q1 | Trend over time | Delay rates peaked in 2007, dropped in COVID, and have rebounded above pre-pandemic levels |
| Q2 | Causes by season | Weather spikes in winter; cascading late-aircraft delays dominate summer |
| Q3 | Carrier rankings | Most carriers are worse post-pandemic than pre-pandemic |
| Q4 | Late-aircraft cascades | Several airports see 50%+ of delays from cascading effects |
| Q5 | Weather vs NAS | Strong correlation — weather-prone airports also suffer NAS overload |
| Q6 | Cancel vs delay trade-off | No reliable trade-off exists; some carriers have both high cancellations and high delays |
| Q7 | Heatmap | Summer + December worst; EWR and JFK chronic across all months |
| Q8 | COVID recovery | Delta recovered fastest; no major carrier is back to 2019 levels |
| Q9 | Self-accountability | Carrier-caused share varies from ~20% to 40%+ across airlines |
| Q10 | Volume vs delay | Volume-delay correlation has weakened — mid-size airports now as bad as mega-hubs |